# Task 2: Sentiment Analysis using NLP Pipeline & ML Models

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\athar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\athar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### 1. Data Understanding

In [2]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
print("Shape:", df.shape)
print("Class Distribution:")
print(df['sentiment'].value_counts())
df.sample(5)

Shape: (50000, 2)
Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
17513,"In Halloween, three friends seek an ancient ce...",negative
4580,"This is what I call a ""pre Sci Fi; Sci Fi"" mov...",positive
28477,I picked up this DVD for $4.99. They had put s...,negative
43276,I saw this when I was twelve. It was the movie...,positive
25024,"Yes, I loved this movie when I was a kid. When...",positive


### 2. NLP Preprocessing

In [4]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    
    words = word_tokenize(text)
    words = [w for w in words if w not in stop_words]
    words = [stemmer.stem(w) for w in words]
    
    return " ".join(words)

In [5]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\athar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
df['clean_text'] = df['review'].apply(preprocess_text)
df.head()

,review,sentiment,clean_text
0,One of the other reviewers has mentioned that ...,positive,one review mention watch 1 oz episod youll hoo...
1,A wonderful little production. <br /><br />The...,positive,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,positive,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,negative,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter mattei love time money visual stun film...


In [7]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

### Train-Test split

In [8]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 3. Feature Engineering

In [9]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

### 4. Model Building

In [10]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [11]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [12]:
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


### 5. Model Evaluation

In [13]:
def evaluate(model):
    y_pred = model.predict(X_test_tfidf)
    
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("----------------------")

### 6. Comparison & Insights

In [14]:
print("Logistic Regression")
evaluate(lr)

print("Naive Bayes")
evaluate(nb)

print("Decision Tree")
evaluate(dt)

Logistic Regression
Accuracy: 0.8861
Precision: 0.8747117601844735
Recall: 0.9033538400476285
F1 Score: 0.8888021087572
----------------------
Naive Bayes
Accuracy: 0.8503
Precision: 0.8467110415035238
Recall: 0.8583052192895416
F1 Score: 0.8524687099635361
----------------------
Decision Tree
Accuracy: 0.7205
Precision: 0.7301066447908121
Recall: 0.7064893828140504
F1 Score: 0.7181038830055472
----------------------


- Logistic Regression is the best model for this dataset.

### Sample Prediction

In [15]:
sample = ["This movie is amazing"]

sample_clean = [preprocess_text(text) for text in sample]
sample_vector = tfidf.transform(sample_clean)

prediction = lr.predict(sample_vector)

print("Result:", "Positive" if prediction[0]==1 else "Negative")

Result: Positive


### Summary of Findings

- Text data was cleaned using preprocessing techniques like lowercasing, stopword removal, and stemming.
- TF-IDF was used for feature extraction and performed effectively.
- Logistic Regression achieved the highest accuracy among all models.
- Naive Bayes performed well with faster execution.
- Decision Tree showed lower accuracy and tendency to overfit.

### Final Conclusion:
- Logistic Regression with TF-IDF is the best combination for this sentiment analysis task.